# Lipkin–Meshkov–Glick Model Coupled to a Cavity
## Photon Number Fluctuations and Spin Noise Spectral Density

### Physical Model

The Hamiltonian couples $N$ collective spins (LMG model) to a single cavity
mode via an $S_z$ interaction, with a periodic transverse-field drive:

$$H(t) = -\frac{J}{N}S_z^2 + h_0\cos(\Omega t)\,S_x
+ \omega_0\,a^\dagger a + \frac{2g}{\sqrt{N}}(a+a^\dagger)\,S_z$$

**Heisenberg EOM for the cavity** (from $\dot{a}=-i[a,H]$):

$$\dot{a} = -i\omega_0\,a - i\frac{2g}{\sqrt{N}}\,S_z(t)$$

The cavity is driven entirely by spin noise $S_z(t)$.  
Whether it thermalises depends on the **spin noise spectral density** at $\omega_0$:

$$S_{zz}(\omega)\approx\left|\mathcal{F}[\langle S_z(t)\rangle]\right|^2$$

| Spin-bath regime | $S_{zz}(\omega)$ shape | Cavity outcome |
|---|---|---|
| Ergodic / chaotic  | broad, featureless | thermalises efficiently |
| Integrable / regular | sharp peaks only | stays near vacuum |

The driven LMG bath enters the **chaotic regime** when $h_0\gtrsim J$
and $\Omega\sim J$ (near-resonant, strong drive).

In [ ]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'font.size': 12})

In [ ]:
def build_ops(N, N_ph, J, omega_0, g):
    """Build operators for the LMG-cavity system.

    Static Hamiltonian:
        H_static = -(J/N) Sz^2  +  omega_0 * a†a  +  (2g/sqrt(N)) * (a+a†) * Sz

    Drive term  h0*cos(Omega*t)*Sx  is added as a time-dependent term.

    Returns
    -------
    H_static, H_drive, n_ph_op, Sz_op, psi0
    """
    S        = N / 2.0
    dim_spin = int(2 * S + 1)

    # Collective spin operators
    Sx     = jmat(S, 'x')
    Sz     = jmat(S, 'z')
    I_spin = qeye(dim_spin)

    # Cavity operators
    a    = destroy(N_ph)
    I_ph = qeye(N_ph)

    # Promote to full Hilbert space  spin ⊗ photon
    Sx_full   = tensor(Sx, I_ph)
    Sz_full   = tensor(Sz, I_ph)
    n_ph_full = tensor(I_spin, a.dag() * a)
    X_ph_full = tensor(I_spin, a + a.dag())    # cavity quadrature

    # Hamiltonian parts
    H_lmg    = -(J / N) * tensor(Sz * Sz, I_ph)              # LMG interaction
    H_cav    = omega_0 * n_ph_full                            # cavity energy
    H_int    = (2.0 * g / np.sqrt(N)) * X_ph_full * Sz_full  # Sz coupling

    H_static = H_lmg + H_cav + H_int
    H_drive  = Sx_full                          # transverse drive operator

    # Initial state: max-Sx eigenstate  x  photon vacuum
    evals, evecs = Sx.eigenstates()
    psi0 = tensor(evecs[np.argmax(evals)], basis(N_ph, 0))

    return H_static, H_drive, n_ph_full, Sz_full, psi0

In [ ]:
# =====================================================================
# CONTROLLABLE PARAMETERS — modify these freely
# =====================================================================
N    = 8       # number of spins  (collective spin S = N/2)
N_ph = 15      # photon Fock-space cutoff  (raise if <n> → N_ph)

J    = 1.0     # LMG coupling  (sets the energy scale)

# --- Drive: CHAOTIC regime when h0 > J and Omega ~ J ---
h0    = 2.5    # drive amplitude   (chaotic: h0 / J ~ 2-3)
Omega = 2.0    # drive frequency   (chaotic: near-resonant with spin gap)

# --- Cavity ---
omega_0 = 1.0  # cavity frequency
g       = 0.10 # spin-cavity coupling  (probe limit: g << J)

# --- Time grid ---
T_total = 300.0  # total evolution time  [units of 1/J]
dt      = 0.05   # time step
# =====================================================================

In [ ]:
def drive_coeff(t, args):
    return args['h0'] * np.cos(args['Omega'] * t)

tlist = np.linspace(0, T_total, int(T_total / dt) + 1)
args  = {'h0': h0, 'Omega': Omega}

H_static, H_drive, n_ph_op, Sz_op, psi0 = build_ops(N, N_ph, J, omega_0, g)
H_td = [H_static, [H_drive, drive_coeff]]

print(f'Hilbert space : {H_static.shape[0]}  ({N+1} spin x {N_ph} photon)')
print(f'Time steps    : {len(tlist)}')
print(f'Drive         : h0={h0}, Omega={Omega}  (h0/J = {h0/J:.1f})')
print('Running sesolve …')

result = mesolve(H_td, psi0, tlist, [], [n_ph_op, Sz_op], args=args)

n_t  = np.array(result.expect[0])   # <a†a>(t)
Sz_t = np.array(result.expect[1])   # <Sz>(t)

if n_t.max() > 0.8 * N_ph:
    print(f'WARNING: max <n> = {n_t.max():.2f} approaches N_ph={N_ph}. Increase N_ph.')
print(f'Done.  max <n> = {n_t.max():.3f},  mean <n> = {n_t.mean():.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(tlist, n_t, lw=0.8, color='steelblue')
ax1.axhline(n_t.mean(), color='tomato', lw=1.5, ls='--',
            label=f'time-average = {n_t.mean():.4f}')
ax1.set_ylabel(r'$\langle \hat{n}\rangle(t)$')
ax1.set_title(
    f'Photon Number  (N={N}, N_ph={N_ph}, g={g}, h0={h0}, Omega={Omega}, omega_0={omega_0})')
ax1.legend()
ax1.grid(alpha=0.3)

delta_n = n_t - n_t.mean()
ax2.plot(tlist, delta_n, lw=0.8, color='darkorange')
ax2.axhline(0, color='gray', lw=0.8, ls=':')
ax2.set_xlabel('Time  [1/J]')
ax2.set_ylabel(r'$\delta n(t)$')
ax2.set_title('Photon number fluctuations around time-average')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('photon_fluctuations.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'RMS photon fluctuation std(n) = {delta_n.std():.4f}')

In [ ]:
# Discard first half (transient); analyse steady-state portion
half   = len(tlist) // 2
Sz_ss  = Sz_t[half:]
Sz_ac  = Sz_ss - Sz_ss.mean()              # remove DC
N_fft  = len(Sz_ac)
freqs  = np.fft.rfftfreq(N_fft, d=dt)
omegas = 2 * np.pi * freqs
S_zz   = np.abs(np.fft.rfft(Sz_ac))**2 / (N_fft * dt)   # one-sided PSD

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))

ax1.plot(tlist[half:], Sz_ss, lw=0.7, color='teal')
ax1.set_xlabel('Time  [1/J]')
ax1.set_ylabel(r'$\langle S_z\rangle$')
ax1.set_title(f'Spin $S_z$ time series (steady-state half,  h0={h0}, Omega={Omega})')
ax1.grid(alpha=0.3)

mask = omegas <= 5 * Omega
ax2.semilogy(omegas[mask], S_zz[mask], lw=1.0, color='purple',
             label=r'$S_{zz}(\omega)$')
ax2.axvline(Omega,   color='red',   ls='--', lw=1.2,
            label=f'drive  Omega={Omega}')
ax2.axvline(omega_0, color='green', ls=':',  lw=1.5,
            label=f'cavity omega_0={omega_0}')
ax2.set_xlabel(r'$\omega$  [rad/J]')
ax2.set_ylabel(r'$S_{zz}(\omega)$  [arb.]')
ax2.set_title('Spin Noise Power Spectral Density')
ax2.legend()
ax2.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('spin_spectral_density.png', dpi=100, bbox_inches='tight')
plt.show()

idx_cav = np.argmin(np.abs(omegas - omega_0))
print(f'Spectral weight at cavity freq omega_0={omega_0}:  S_zz = {S_zz[idx_cav]:.3e}')

## Comparison: Chaotic vs Integrable Spin Bath

With identical cavity/coupling parameters, we contrast two drive regimes:

- **Chaotic**: strong drive $h_0=2.5J$, near-resonant $\Omega=2.0J$
- **Integrable**: weak drive $h_0=0.3J$, same $\Omega$

The chaotic bath produces a **broad** $S_{zz}(\omega)$, meaning significant
spectral weight at all frequencies including $\omega_0$, leading to larger
photon number fluctuations.  
The integrable bath produces **sharp harmonic peaks** and the cavity is
barely excited (small $\sigma_n$).

In [ ]:
param_sets = [
    dict(h0=2.5, Omega=2.0, label='Chaotic   (h0=2.5, Omega=2.0)', color='purple'),
    dict(h0=0.3, Omega=2.0, label='Integrable (h0=0.3, Omega=2.0)', color='darkorange'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, p in enumerate(param_sets):
    res_i = mesolve(
        [H_static, [H_drive, drive_coeff]],
        psi0, tlist, [],
        [n_ph_op, Sz_op],
        args={'h0': p['h0'], 'Omega': p['Omega']}
    )
    n_i  = np.array(res_i.expect[0])
    Sz_i = np.array(res_i.expect[1])

    half_i   = len(tlist) // 2
    Sz_ac_i  = Sz_i[half_i:] - Sz_i[half_i:].mean()
    N_fi     = len(Sz_ac_i)
    omegas_i = 2 * np.pi * np.fft.rfftfreq(N_fi, d=dt)
    S_zz_i   = np.abs(np.fft.rfft(Sz_ac_i))**2 / (N_fi * dt)
    mask_i   = omegas_i <= 5 * p['Omega']

    # Top row: photon number
    axes[0, col].plot(tlist, n_i, lw=0.7, color=p['color'])
    axes[0, col].axhline(n_i.mean(), color='gray', ls='--', lw=1.0,
                         label=f'mean={n_i.mean():.4f}')
    axes[0, col].set_title(p['label'])
    axes[0, col].set_xlabel('Time [1/J]')
    axes[0, col].set_ylabel(r'$\langle\hat{n}\rangle$')
    axes[0, col].legend(fontsize=9)
    axes[0, col].grid(alpha=0.3)

    # Bottom row: spin spectral density
    axes[1, col].semilogy(omegas_i[mask_i], S_zz_i[mask_i],
                          lw=1.0, color=p['color'])
    axes[1, col].axvline(omega_0,    color='green', ls=':',  lw=1.5,
                         label=f'omega_0={omega_0}')
    axes[1, col].axvline(p['Omega'], color='red',   ls='--', lw=1.2,
                         label=f"Omega={p['Omega']}")
    axes[1, col].set_xlabel(r'$\omega$  [rad/J]')
    axes[1, col].set_ylabel(r'$S_{zz}(\omega)$')
    axes[1, col].legend(fontsize=9)
    axes[1, col].grid(alpha=0.3, which='both')

    idx = np.argmin(np.abs(omegas_i - omega_0))
    sig_n = n_i.std()
    print(f'{p["label"]:42s}  S_zz(omega_0)={S_zz_i[idx]:.3e}  '
          f'max<n>={n_i.max():.4f}  std(n)={sig_n:.4f}')

fig.suptitle(
    f'LMG-cavity: N={N}, N_ph={N_ph}, g={g}, omega_0={omega_0}\n'
    'Chaotic (broad spectrum) vs Integrable (sharp peaks)',
    fontsize=12
)
plt.tight_layout()
plt.savefig('comparison_chaotic_vs_integrable.png', dpi=100, bbox_inches='tight')
plt.show()